# Case Study — PV Load-Curve Forecasting & Segmentation
**Context:** Master's thesis (Complutense) · **Synthetic data for portfolio demo**

Goal: cluster customers by load shape and evaluate a simple day-ahead forecast baseline.


In [ ]:
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_percentage_error

rng = np.random.default_rng(11)
hours = np.arange(24)
n_customers = 240

profiles = []
for cid in range(n_customers):
    kind = rng.choice(["residential", "commercial", "mixed"], p=[0.5, 0.3, 0.2])
    if kind == "residential":
        curve = 0.4 + 0.35 * np.exp(-0.5 * ((hours - 7) / 2.2) ** 2) + 0.55 * np.exp(-0.5 * ((hours - 20) / 2.5) ** 2)
    elif kind == "commercial":
        curve = 0.25 + 0.9 * ((hours >= 8) & (hours <= 18)).astype(float)
    else:
        curve = 0.35 + 0.45 * np.sin((hours / 24) * 2 * np.pi) + 0.25
    curve = curve * rng.uniform(0.8, 1.25) + rng.normal(0, 0.03, 24)
    profiles.append(np.clip(curve, 0.05, None))

load = pd.DataFrame(profiles, columns=[f"h{h:02d}" for h in hours])
load["customer_id"] = [f"PV{i:03d}" for i in range(n_customers)]
load.head()


In [ ]:
# Segmentation on normalized daily shape
X = load.filter(like="h")
X_norm = X.div(X.sum(axis=1), axis=0)
scaler = StandardScaler()
Z = scaler.fit_transform(X_norm)
km = KMeans(n_clusters=3, n_init=10, random_state=11)
load["segment"] = km.fit_predict(Z)
load["segment"].value_counts().sort_index()


In [ ]:
# Day-ahead naive forecast proxy on aggregate curve
agg = X.mean(axis=0).values
# Simulate 14 days of noisy aggregate demand
series = []
for d in range(14):
    series.append(agg * rng.uniform(0.92, 1.08) + rng.normal(0, 0.02, 24))
series = np.array(series)
y_true = series[1:].ravel()
y_pred = series[:-1].ravel()  # yesterday's shape as forecast
mape = mean_absolute_percentage_error(y_true, y_pred)
{
    "segments": int(load["segment"].nunique()),
    "baseline_mape": round(float(mape), 3),
    "peak_hour": int(agg.argmax()),
}
